In [ ]:
from datetime import datetime
 
from pyspark.sql import functions as F
from pyspark.sql.window import Window
 
from ssv_data.io.writers import write_delta
from ssv_data.schema.cast import fill_missing_columns, cast_by_schema, select_schema
from ssv_data.transforms.common import range_join_effective, coalesce_sources, add_audit_columns
from ssv_data.transforms.scd import as_of_join

In [ ]:
# Full fact column set (matches the original df_merge_all_schema, plus the columns
# main() appended: customer_phone, put_in_cart_order, feedback_at).
FACT_SCHEMA = {
    "transaction_id": "str", "product_id": "str", "report_date": "date", "transaction_time": "date",
    "product_name": "str", "product_uom": "str",
    "product_group_id": "int", "product_group_name": "str",
    "product_sub_category_id": "int", "product_sub_category_name": "str",
    "product_category_id": "int", "product_category_name": "str",
    "product_price": "float", "product_brand_name": "str", "product_manufacturer_name": "str",
    "retail_business_type": "str",
    "store_id": "str", "store_name": "str", "store_type": "str",
    "supplier_id": "str", "supplier_name": "str",
    "product_uom_size": "int", "quantity": "int",
    "total_amount_no_tax": "float", "total_amount": "float",
    "final_amount_no_tax": "float", "final_amount": "float",
    "row_num_transaction_id": "int", "sale_bill_time": "date", "posting_time": "date",
    "staff_id": "str", "staff_name": "str",
    "payment_method_id": "int", "payment_method_name": "str",
    "payment_supplier_id": "int", "payment_supplier_name": "str",
    "is_passive_capture": "int", "order_no": "str",
    "bill_amount": "float", "voucher_amount": "float",
    "customer_id": "int", "customer_phone": "str", "external_order_id": "str",
    "dom_created_at": "date", "partner_store": "str",
    "user_name": "str", "customer_code": "str", "channel_name": "str",
    "rating": "int", "comment": "str", "problem_label_by_llm": "str",
    "delivery_created_time": "date", "delivery_driver_booked_time": "date",
    "delivery_picked_time": "date", "delivery_canceled_time": "date",
    "delivery_completed_time": "date", "delivery_status": "str",
    "delivery_partial_completed_time": "date", "delivery_store_delivered_time": "date",
    "order_id": "int", "purchase_price_without_tax": "float", "purchase_price_with_tax": "float",
    "order_cancel_reason": "str",
    "promotion_id_arr": "list", "promotion_name_arr": "list",
    "voucher_code_arr": "list", "voucher_name_arr": "list",
    "number_of_line_item": "int", "total_voucher_amount": "float", "capture_method": "str",
    "customer_gender": "str", "customer_age_range": "str", "customer_nationality": "str",
    "synced_sap_time": "date", "refund_transaction_id": "str", "product_uom_id": "int",
    "transaction_type": "str", "delivery_note": "str", "short_order_id": "str",
    "output_vat_code": "str", "transaction_7now_source": "str",
    "put_in_cart_order": "int", "feedback_at": "date",
}

# Delivery enrichment columns carried onto the fact (joined twice: by txn_id and order_no).
DELIV_ENRICH = ["customer_id", "customer_phone", "user_name", "customer_code",
                "external_order_id", "delivery_note", "short_order_id", "order_id",
                "channel_name", "partner_store", "dom_created_at",
                "delivery_created_time", "delivery_driver_booked_time", "delivery_picked_time",
                "delivery_canceled_time", "delivery_completed_time", "delivery_store_delivered_time",
                "delivery_partial_completed_time"]
DELIV_TIMES = ["delivery_created_time", "delivery_driver_booked_time", "delivery_picked_time",
               "delivery_canceled_time", "delivery_completed_time", "delivery_store_delivered_time",
               "delivery_partial_completed_time"]

In [ ]:
def _pfx(df, prefix, key, cols, extra=None):
    sel = [F.col(key)] + [F.col(c).alias(f"{prefix}_{c}") for c in cols]
    if extra:
        sel += extra
    return df.select(*sel)

def _merge(ctx):
    s = ctx.spark
    lines = s.read.table(ctx.silver("sale_line"))
    promo = s.read.table(ctx.silver("promotion"))
    delivery = s.read.table(ctx.silver("delivery"))
    txn = s.read.table(ctx.silver("transaction"))
    pph = s.read.table(ctx.silver("point_history"))
    d_oms = s.read.table(ctx.silver("dim_oms_product"))
    # dim_store is SCD2 — keep the version columns for the as-of join below
    d_store = s.read.table(ctx.silver("dim_store")).select(
        "store_id", "sale_area_id", "valid_from", "valid_to")
    d_price = s.read.table(ctx.silver("dim_purchase_price"))
 
    fact = (lines
            .join(promo, "transaction_id", "left")
            .join(_pfx(txn, "txn", "transaction_id",
                       ["customer_id", "customer_phone", "user_name", "customer_code", "rating", "comment", "dom_created_at"]),
                  "transaction_id", "left")
            .join(_pfx(pph, "pph", "transaction_id",
                       ["customer_id", "customer_phone", "user_name", "customer_code", "capture_method", "dom_created_at"],
                       extra=[F.lit(1).alias("pph_flag")]),
                  "transaction_id", "left")
            .join(_pfx(delivery, "dtx", "transaction_id", DELIV_ENRICH), "transaction_id", "left")
            .join(_pfx(delivery, "dno", "order_no", DELIV_ENRICH), "order_no", "left")
            .join(d_oms, "product_id", "left"))
 
    # store attrs as-of transaction time (SCD2) — a backfill picks the store's
    # sale_area of THAT day. Must precede the price join, which keys on sale_area_id.
    fact = as_of_join(fact, d_store, ["store_id"], ts_col="transaction_time")
 
    # effective-dated purchase price on (product_id, sale_area_id)
    fact = range_join_effective(
        fact, d_price, keys=["product_id", "sale_area_id"],
        ts_col="transaction_time",
        eff_col="purchase_price_effective_date",
        next_col="next_purchase_price_effective_date")
 
    # identity coalesce — customer_id priority flips on 2022-04-01 (matches the original)
    after = datetime.strptime(ctx.running_date, "%Y-%m-%d") > datetime(2022, 4, 1)
    cust = (["pph_customer_id", "txn_customer_id", "dtx_customer_id", "dno_customer_id"] if after
            else ["dtx_customer_id", "dno_customer_id", "pph_customer_id", "txn_customer_id"])
    fact = coalesce_sources(fact, "customer_id", cust)
    fact = coalesce_sources(fact, "customer_phone", ["dtx_customer_phone", "dno_customer_phone", "pph_customer_phone", "txn_customer_phone"])
    fact = coalesce_sources(fact, "user_name", ["dtx_user_name", "dno_user_name", "pph_user_name", "txn_user_name"])
    fact = coalesce_sources(fact, "customer_code", ["dtx_customer_code", "dno_customer_code", "pph_customer_code", "txn_customer_code"])
    fact = coalesce_sources(fact, "channel_name", ["dtx_channel_name", "dno_channel_name", "channel_name"])
    fact = coalesce_sources(fact, "external_order_id", ["dtx_external_order_id", "dno_external_order_id"])
    fact = coalesce_sources(fact, "delivery_note", ["dtx_delivery_note", "dno_delivery_note"])
    fact = coalesce_sources(fact, "short_order_id", ["dtx_short_order_id", "dno_short_order_id"])
    fact = coalesce_sources(fact, "order_id", ["dtx_order_id", "dno_order_id"])
    fact = coalesce_sources(fact, "dom_created_at", ["dtx_dom_created_at", "dno_dom_created_at", "pph_dom_created_at", "txn_dom_created_at"])
    fact = coalesce_sources(fact, "partner_store", ["dtx_partner_store", "dno_partner_store", "promotion_partner_store"])
    for t in DELIV_TIMES:
        fact = coalesce_sources(fact, t, [f"dno_{t}", f"dtx_{t}"])
 
    return (fact
            .withColumn("rating", F.col("txn_rating"))
            .withColumn("comment", F.col("txn_comment"))
            .withColumn("capture_method", F.col("pph_capture_method"))
            .withColumn("is_passive_capture", F.when(F.col("pph_flag").isNotNull(), F.lit(1)).otherwise(F.lit(0)))
            .withColumn("order_cancel_reason", F.lit(""))
            .withColumn("transaction_7now_source", F.lit("")))

In [ ]:
def _put_in_cart(ctx):
    s = ctx.spark
    completed = (s.read.table(ctx.bronze("delivery_orders")).where(F.col("status") == 8)
                 .select(F.col("order_id"), F.col("transaction_no").alias("transaction_id")))
    return (s.read.table(ctx.bronze("delivery_order_details"))
            .join(completed, "order_id", "inner")
            .where(F.col("put_in_cart_order").isNotNull())
            .select("transaction_id", "product_id", "put_in_cart_order")
            .dropDuplicates(["transaction_id", "product_id"]))

def _canceled(ctx):
    s = ctx.spark
    delivery = s.read.table(ctx.silver("delivery"))
    lines = s.read.table(ctx.silver("sale_line"))
    txn_keys = lines.select(F.col("transaction_id").alias("k_txn")).distinct()
    no_keys = lines.select(F.col("order_no").alias("k_no")).where(F.col("k_no").isNotNull()).distinct()
    d_prod = s.read.table(ctx.silver("dim_product"))
    pm = F.col("payment_method")
 
    canc = (delivery.where(F.col("status").isin(5, 13))
            .join(txn_keys, F.col("transaction_id") == F.col("k_txn"), "left_anti")
            .join(no_keys, F.col("order_no") == F.col("k_no"), "left_anti")
            .join(s.read.table(ctx.bronze("delivery_order_details")).select("order_id", "product_id"), "order_id", "left"))
    # product attrs as-of the order's creation time (dim_product is SCD2)
    canc = as_of_join(
        canc,
        d_prod.select("product_id", "product_name", "product_uom", "product_uom_size", "product_uom_id",
                      "product_group_id", "product_group_name", "product_category_id", "product_category_name",
                      "product_sub_category_id", "product_sub_category_name", "retail_business_type",
                      "valid_from", "valid_to"),
        ["product_id"], ts_col="dom_created_at")
    canc = (canc
            .withColumn("report_date", F.to_date("dom_created_at"))
            .withColumn("transaction_time", F.col("dom_created_at"))
            .withColumn("delivery_status", F.lit("canceled"))
            .withColumn("transaction_type", F.lit("Sale Transaction"))
            .withColumn("final_amount", F.col("total_price_amount"))
            .withColumn("final_amount_no_tax", F.col("total_price_amount"))
            .withColumn("total_amount", F.col("total_price_amount"))
            .withColumn("total_amount_no_tax", F.col("total_price_amount"))
            .withColumnRenamed("total_price_amount", "bill_amount")
            .withColumnRenamed("canceled_reason", "order_cancel_reason")
            .withColumn("payment_method_name",
                        F.when(pm == 0, "cash").when(pm == 1, "card").when(pm == 2, "voucher").when(pm == 4, "e-wallet").otherwise("unknown"))
            .withColumnRenamed("payment_method", "payment_method_id")
            .withColumn("row_num_transaction_id", F.row_number().over(Window.partitionBy("order_id").orderBy(F.monotonically_increasing_id())))
            .withColumn("number_of_line_item", F.count("*").over(Window.partitionBy("order_id")))
            .withColumn("refund_transaction_id", F.lit(""))
            .withColumn("transaction_7now_source", F.lit("")))
    # TODO: product upc-vs-hq fallback for canceled lines (product_uom_upc path).
    return canc

In [ ]:
def _finalize(ctx, df, is_canceled=False):
    df = df.withColumn("feedback_at", F.to_timestamp(F.lit("1971-01-01")))
    if is_canceled:
        df = df.withColumn("put_in_cart_order", F.lit(0))
    df = fill_missing_columns(df, FACT_SCHEMA)
    df = cast_by_schema(df, FACT_SCHEMA)
    df = df.withColumn("report_date", F.to_date("report_date"))  # DateType for clean partition values
    return add_audit_columns(select_schema(df, FACT_SCHEMA))
 
 
def gold_build(ctx) -> None:
    fact = _merge(ctx).join(_put_in_cart(ctx), ["transaction_id", "product_id"], "left")
    fact = fact.withColumn("put_in_cart_order", F.coalesce(F.col("put_in_cart_order"), F.lit(0)))
 
    gold = _finalize(ctx, fact).unionByName(_finalize(ctx, _canceled(ctx), is_canceled=True),
                                            allowMissingColumns=True)
    # Keep only the run-day partition so the write matches replaceWhere exactly.
    gold = gold.where(F.col("report_date") == F.to_date(F.lit(ctx.running_date)))
    # replaceWhere on report_date == the old DELETE WHERE transaction_time ... + INSERT, atomically.
    # Boundary: to_date(transaction_time) must equal report_date for both sale & canceled rows.
    write_delta(ctx, gold, ctx.gold("fact_eod_sale_product"),
                partition_col="report_date", replace_where=ctx.replace_where)
    ctx.logger.info("gold build complete")